# PDF Input Quickstart for Qwen3.5-27B on vLLM

この Notebook は、英語論文 PDF の **全ページ** を対象にして、vLLM の OpenAI 互換 API 経由で **text として入れる方法** と **画像化して vision input として入れる方法** をまとめたものです。


## 1. 前提
- Docker で vLLM API サーバが起動している
- `OPENAI_BASE_URL` を必要に応じて設定する
- PDF サンプルは `papers/` に配置済み


In [1]:
import os
from pathlib import Path
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:8000/v1')
PAPERS = Path('../papers')
print('Using base_url =', BASE_URL)
print('Papers:', sorted(p.name for p in PAPERS.glob('*.pdf')))
client = OpenAI(api_key='EMPTY', base_url=BASE_URL, timeout=3600)
client


Using base_url = http://127.0.0.1:30010/v1
Papers: ['attention_is_all_you_need.pdf', 'deep_residual_learning.pdf']


## 2. PDF の全ページから text を抽出して入れる


In [2]:
from pypdf import PdfReader

pdf_path = PAPERS / 'attention_is_all_you_need.pdf'
reader = PdfReader(str(pdf_path))
text = '\n'.join(page.extract_text() or '' for page in reader.pages)
print('page_count =', len(reader.pages))
print('text_chars =', len(text))
print(text[:3000])


page_count = 15
text_chars = 39629
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recur

In [3]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'system', 'content': 'You are reading a full English deep learning paper extracted from PDF text.'},
        {'role': 'user', 'content': 'Summarize the paper in Japanese with sections for problem, method, and key ideas.\n\n' + text[:60000]},
    ],
    max_tokens=512,
)
resp


ChatCompletion(id='chatcmpl-adb95b86a9e24949', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='Here\'s a thinking process that leads to the suggested summary:\n\n1.  **Analyze the Request:**\n    *   **Source Material:** A full English deep learning paper ("Attention Is All You Need") extracted from PDF text.\n    *   **Task:** Summarize the paper in Japanese.\n    *   **Structure:** Must include sections for "Problem" (課題), "Method" (手法), and "Key Ideas" (主要なアイデア).\n    *   **Constraint:** Proper attribution is implied by the prompt\'s context (Google grants permission), but the summary itself needs to reflect the content accurately.\n\n2.  **Analyze the Source Text (Paper Content):**\n    *   **Title:** Attention Is All You Need.\n    *   **Authors:** Vaswani et al. (Google Brain/Research).\n    *   **Abstract:** Propo

## 3. PDF の全ページを画像化して入れる


In [4]:
import fitz  # pymupdf

pdf_path = PAPERS / 'deep_residual_learning.pdf'
doc = fitz.open(pdf_path)
out_dir = PAPERS / 'rendered' / 'deep_residual_learning_all_pages_vllm'
out_dir.mkdir(parents=True, exist_ok=True)
image_paths = []
for i in range(len(doc)):
    page = doc.load_page(i)
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
    img_path = out_dir / f'deep_residual_learning_page_{i+1:02d}.png'
    pix.save(img_path)
    image_paths.append(img_path)
print('rendered_pages =', len(image_paths))
print(image_paths[:3], '...')


rendered_pages = 12
[PosixPath('../papers/rendered/deep_residual_learning_all_pages_vllm/deep_residual_learning_page_01.png'), PosixPath('../papers/rendered/deep_residual_learning_all_pages_vllm/deep_residual_learning_page_02.png'), PosixPath('../papers/rendered/deep_residual_learning_all_pages_vllm/deep_residual_learning_page_03.png')] ...


In [5]:
import base64
import mimetypes

content = [{'type': 'text', 'text': 'These are rendered pages from an English deep learning paper PDF. Summarize the paper in Japanese and mention the overall topic and architecture.'}]
for img_path in image_paths:
    mime = mimetypes.guess_type(img_path.name)[0] or 'image/png'
    image_url = 'data:' + mime + ';base64,' + base64.b64encode(img_path.read_bytes()).decode('utf-8')
    content.append({'type': 'image_url', 'image_url': {'url': image_url}})

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[{'role': 'user', 'content': content}],
    max_tokens=512,
)
resp


ChatCompletion(id='chatcmpl-bf288696f302f06b', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='The user wants a summary of the provided research paper in Japanese.\nThe paper is "Deep Residual Learning for Image Recognition" by He et al. (Microsoft Research).\n\n**Key points to cover:**\n1.  **Overall Topic:** Deep residual learning for image recognition. The core problem is that deeper networks are harder to train (degradation problem), and the solution is residual learning.\n2.  **Architecture:** Residual Neural Networks (ResNets). The key building block is the "residual block" with a shortcut connection (identity mapping).\n    *   Formula: $y = F(x, \\{W_i\\}) + x$\n    *   This allows the network to learn residual functions $F(x) \\approx H(x) - x$ instead of the underlying mapping $H(x)$.\n3.  **Key Findings/Result

## 4. 補足
- text 抽出は本文全体の要約向き
- 画像化は図表・レイアウト・数式を含めた理解向き
